In [1]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], 
[0.55, 0.87, 0.66], 
[0.57, 0.85, 0.64], 
[0.22, 0.58, 0.33], 
[0.77, 0.25, 0.10], 
[0.05, 0.80, 0.55]] 
)

In [2]:
import torch.nn as nn
class SelfAttentionWithLinearLayers(nn.Module):
    def __init__(self, d_in, d_out,qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax( attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [3]:
d_in = inputs.shape[1]
d_out = 2
SaLL = SelfAttentionWithLinearLayers(d_in,d_out)

In [5]:
queries = SaLL.W_query(inputs)
keys = SaLL.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1811, 0.1628, 0.1627, 0.1648, 0.1628, 0.1658],
        [0.1930, 0.1598, 0.1596, 0.1631, 0.1594, 0.1651],
        [0.1924, 0.1600, 0.1598, 0.1632, 0.1593, 0.1653],
        [0.1821, 0.1624, 0.1623, 0.1646, 0.1631, 0.1654],
        [0.1745, 0.1660, 0.1656, 0.1655, 0.1593, 0.1691],
        [0.1887, 0.1600, 0.1600, 0.1638, 0.1640, 0.1635]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [7]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1811, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1930, 0.1598, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1924, 0.1600, 0.1598, 0.0000, 0.0000, 0.0000],
        [0.1821, 0.1624, 0.1623, 0.1646, 0.0000, 0.0000],
        [0.1745, 0.1660, 0.1656, 0.1655, 0.1593, 0.0000],
        [0.1887, 0.1600, 0.1600, 0.1638, 0.1640, 0.1635]],
       grad_fn=<MulBackward0>)


In [8]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5470, 0.4530, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3756, 0.3125, 0.3120, 0.0000, 0.0000, 0.0000],
        [0.2712, 0.2419, 0.2417, 0.2452, 0.0000, 0.0000],
        [0.2100, 0.1998, 0.1993, 0.1992, 0.1917, 0.0000],
        [0.1887, 0.1600, 0.1600, 0.1638, 0.1640, 0.1635]],
       grad_fn=<DivBackward0>)


### an approach for softmax function using -infinity values above the diagonal different from the previous approach. which is easier


In [9]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[ 0.0796,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.1430, -0.1236,    -inf,    -inf,    -inf,    -inf],
        [ 0.1414, -0.1187, -0.1208,    -inf,    -inf,    -inf],
        [ 0.0817, -0.0802, -0.0811, -0.0610,    -inf,    -inf],
        [ 0.0730,  0.0024, -0.0007, -0.0014, -0.0557,    -inf],
        [ 0.1021, -0.1313, -0.1314, -0.0984, -0.0967, -0.1003]],
       grad_fn=<MaskedFillBackward0>)


In [10]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5470, 0.4530, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3756, 0.3125, 0.3120, 0.0000, 0.0000, 0.0000],
        [0.2712, 0.2419, 0.2417, 0.2452, 0.0000, 0.0000],
        [0.2100, 0.1998, 0.1993, 0.1992, 0.1917, 0.0000],
        [0.1887, 0.1600, 0.1600, 0.1638, 0.1640, 0.1635]],
       grad_fn=<SoftmaxBackward0>)


Casual attention class

In [11]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
        dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length),
        diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

In [19]:
torch.manual_seed(122)
context_length = batch.shape[1]
ca1 = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca1(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [20]:
context_vecs

tensor([[[ 0.3422, -0.2407],
         [ 0.3464, -0.2191],
         [ 0.3412, -0.2036],
         [ 0.3110, -0.1928],
         [ 0.2272, -0.0867],
         [ 0.2567, -0.1375]],

        [[ 0.3422, -0.2407],
         [ 0.3464, -0.2191],
         [ 0.3412, -0.2036],
         [ 0.3110, -0.1928],
         [ 0.2272, -0.0867],
         [ 0.2567, -0.1375]]], grad_fn=<UnsafeViewBackward0>)